<a href="https://colab.research.google.com/github/maewhyxng2-prog/zyrquen-platform/blob/main/tools/notebooks/tpu_compatibility_suite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZYRQUEN — Colab TPU / TorchTPU Compatibility & Readiness Suite

ชุดตรวจสอบและเตรียมพร้อมสำหรับ **Google Colab TorchTPU (`v5e-1 TPU` / `v6e-1 TPU`)** และ **PyTorch XLA** โดยไม่แตะต้องหรือเขียนทับ Checkpoint เดิม (เช่น Step 6000 หรือ Step 6600)

### ลำดับการใช้งาน
1. **Runtime → Change runtime type → `v5e-1 TPU` หรือ `v6e-1 TPU`** แล้วกด Save
2. รัน **Cell 1** เพื่อตั้งค่า `REPO_DIR` และตรวจสอบ Accelerator / `COLAB_TPU_ADDR` / `torch.tpu` / `torch_xla`
3. รัน **Cell 2** เพื่อรัน `tools/tpu_suite.py` และ unit tests ของ `tpu_backend`
4. รัน **Cell 3** เพื่อตรวจความสมบูรณ์ของ Checkpoint ล่าสุด (`6000` หรือ `6600`) ในโหมดอ่านอย่างเดียว (`verify-only`)

In [ ]:
#@title 1. ตั้งค่า Repo และตรวจ Colab TPU / TorchTPU Hardware
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/zyrquen-platform")
REPO_URL = "https://github.com/maewhyxng2-prog/zyrquen-platform.git"

if not REPO_DIR.exists():
    if (Path.cwd() / "python_core" / "v2" / "tpu_backend.py").exists():
        REPO_DIR = Path.cwd()
    elif (Path.cwd().parent.parent / "python_core" / "v2" / "tpu_backend.py").exists():
        REPO_DIR = Path.cwd().parent.parent
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

for p in (str(REPO_DIR / "python_core"), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from tools.tpu_probe import detect_runtime_accelerator, format_runtime_report

runtime_info = detect_runtime_accelerator()
print(format_runtime_report(runtime_info))

In [ ]:
#@title 2. รันชุดทดสอบ TPU Compatibility Suite (Non-Destructive)
import unittest
from tools.tpu_suite import run_pure_native_smoke, run_torch_tpu_smoke

pure_smoke = run_pure_native_smoke()
torch_smoke = run_torch_tpu_smoke(runtime_info)
print("Pure Native V2 Smoke :", pure_smoke)
print("Torch/TPU Smoke      :", torch_smoke)

suite = unittest.defaultTestLoader.discover(str(REPO_DIR / "python_core" / "tests"), pattern="test_tpu_backend.py")
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "TPU backend unit tests failed"

In [ ]:
#@title 3. ตรวจ Checkpoint ล่าสุดในโหมดอ่านอย่างเดียว (ไม่แตะต้องหรือเขียนทับ)
from tools.resume_training import DEFAULT_OUT_DIR, resume_from_checkpoint

STEP_TO_VERIFY = 6000  #@param {type:"integer"}
if not DEFAULT_OUT_DIR.is_dir():
    print(f"[INFO] ยังไม่พบ {DEFAULT_OUT_DIR} ในเครื่องนี้ — ข้ามการตรวจโหลด checkpoint")
else:
    restored = resume_from_checkpoint(step=STEP_TO_VERIFY, out_dir=DEFAULT_OUT_DIR)
    print(
        f"VERIFIED step={restored['global_step']} | sha256={restored['sha256']} | "
        f"can_continue_safely={restored['can_continue_safely']}"
    )